# Market Response Analysis - Multi-Asset, Cross-Regional

This notebook analyzes how different asset classes and regions respond to Strait of Hormuz disruptions.

## Analysis Scope:
- **Asset Classes**: Equities, Fixed Income, Commodities, Currencies
- **Regions**: USA, Europe, Asia
- **Time Periods**: Normal vs. Crisis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_acquisition import DataAcquisition
from shipping_monitor import ShippingMonitor
from geopolitical_scorer import GeopoliticalScorer
import yaml

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 10)

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

## 1. Load All Data

In [ ]:
# Fetch data
acq = DataAcquisition()
data = acq.fetch_all_data()

# Analyze shipping and geopolitical risk
monitor = ShippingMonitor()
shipping_analyzed = monitor.detect_anomalies(data['shipping'])

scorer = GeopoliticalScorer()
geo_analyzed = scorer.classify_risk_level(data['geopolitical'])

print(f"Data loaded: {len(shipping_analyzed)} days")
print(f"Crisis days: {(shipping_analyzed['alert_level'] == 'crisis').sum()}")

## 2. Regional Equity Market Response

In [ ]:
# Calculate returns for regional indices
equity_data = data['market']['equities']

# Combine all equity data
all_equities = pd.DataFrame()
for region, df in equity_data.items():
    if isinstance(df, pd.DataFrame):
        for col in df.columns:
            all_equities[f"{region}_{col}"] = df[col]

# Calculate daily returns
equity_returns = all_equities.pct_change()

# Merge with crisis indicator
crisis_indicator = (shipping_analyzed['alert_level'] == 'crisis').astype(int)
analysis_df = equity_returns.copy()
analysis_df['is_crisis'] = crisis_indicator

# Compare returns: Normal vs Crisis
normal_returns = analysis_df[analysis_df['is_crisis'] == 0].drop('is_crisis', axis=1).mean() * 252 * 100
crisis_returns = analysis_df[analysis_df['is_crisis'] == 1].drop('is_crisis', axis=1).mean() * 252 * 100

comparison = pd.DataFrame({
    'Normal (% ann.)': normal_returns,
    'Crisis (% ann.)': crisis_returns,
    'Difference': crisis_returns - normal_returns
})

print("\nEquity Returns: Normal vs Crisis Periods")
print(comparison.round(2))

In [ ]:
# Visualize regional equity performance
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Plot 1: Returns comparison
comparison[['Normal (% ann.)', 'Crisis (% ann.)']].plot(kind='bar', ax=axes[0, 0], 
                                                          color=['green', 'red'], alpha=0.7)
axes[0, 0].set_title('Annualized Returns: Normal vs Crisis', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Return (%)')
axes[0, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 0].legend(['Normal', 'Crisis'])
axes[0, 0].grid(True, alpha=0.3)
plt.setp(axes[0, 0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Plot 2: Return difference
comparison['Difference'].plot(kind='bar', ax=axes[0, 1], color='orange', alpha=0.7)
axes[0, 1].set_title('Crisis Impact (Difference in Returns)', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Return Difference (%)')
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 1].grid(True, alpha=0.3)
plt.setp(axes[0, 1].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Plot 3: Volatility comparison
normal_vol = analysis_df[analysis_df['is_crisis'] == 0].drop('is_crisis', axis=1).std() * np.sqrt(252) * 100
crisis_vol = analysis_df[analysis_df['is_crisis'] == 1].drop('is_crisis', axis=1).std() * np.sqrt(252) * 100

vol_comparison = pd.DataFrame({
    'Normal': normal_vol,
    'Crisis': crisis_vol
})

vol_comparison.plot(kind='bar', ax=axes[1, 0], color=['blue', 'darkred'], alpha=0.7)
axes[1, 0].set_title('Annualized Volatility: Normal vs Crisis', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Volatility (%)')
axes[1, 0].legend(['Normal', 'Crisis'])
axes[1, 0].grid(True, alpha=0.3)
plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Plot 4: Sharpe ratio comparison
normal_sharpe = normal_returns / normal_vol
crisis_sharpe = crisis_returns / crisis_vol

sharpe_comparison = pd.DataFrame({
    'Normal': normal_sharpe,
    'Crisis': crisis_sharpe
})

sharpe_comparison.plot(kind='bar', ax=axes[1, 1], color=['green', 'red'], alpha=0.7)
axes[1, 1].set_title('Sharpe Ratio: Normal vs Crisis', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Sharpe Ratio')
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1, 1].legend(['Normal', 'Crisis'])
axes[1, 1].grid(True, alpha=0.3)
plt.setp(axes[1, 1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 3. Asset Class Performance During Crises

In [ ]:
# Analyze different asset classes
asset_performance = {}

# Fixed Income
if 'fixed_income' in data['market']:
    fi_data = data['market']['fixed_income']
    fi_returns = fi_data.pct_change()
    fi_returns['is_crisis'] = crisis_indicator
    
    asset_performance['Treasuries (TLT)'] = {
        'normal': fi_returns[fi_returns['is_crisis'] == 0]['TLT'].mean() * 252 * 100,
        'crisis': fi_returns[fi_returns['is_crisis'] == 1]['TLT'].mean() * 252 * 100
    }

# Commodities
if 'commodities' in data['market']:
    comm_data = data['market']['commodities']
    comm_returns = comm_data.pct_change()
    comm_returns['is_crisis'] = crisis_indicator
    
    for ticker in ['USO', 'BNO', 'UNG', 'BDRY']:
        if ticker in comm_returns.columns:
            asset_performance[f'{ticker}'] = {
                'normal': comm_returns[comm_returns['is_crisis'] == 0][ticker].mean() * 252 * 100,
                'crisis': comm_returns[comm_returns['is_crisis'] == 1][ticker].mean() * 252 * 100
            }

# Create comparison dataframe
asset_comp = pd.DataFrame(asset_performance).T
asset_comp['difference'] = asset_comp['crisis'] - asset_comp['normal']

print("\nAsset Class Performance: Normal vs Crisis")
print(asset_comp.round(2))

# Visualize
fig, ax = plt.subplots(figsize=(14, 6))
asset_comp[['normal', 'crisis']].plot(kind='bar', ax=ax, color=['green', 'red'], alpha=0.7)
ax.set_title('Asset Class Returns: Normal vs Crisis Periods', fontsize=16, fontweight='bold')
ax.set_ylabel('Annualized Return (%)')
ax.set_xlabel('Asset')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.legend(['Normal', 'Crisis'])
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Regional Impact Analysis

In [ ]:
# Calculate regional impact
regional_impact = {}

for region in ['us', 'europe', 'asia']:
    region_cols = [col for col in equity_returns.columns if col.startswith(region)]
    if region_cols:
        region_returns = equity_returns[region_cols].mean(axis=1)
        
        normal_ret = region_returns[crisis_indicator == 0].mean() * 252 * 100
        crisis_ret = region_returns[crisis_indicator == 1].mean() * 252 * 100
        normal_vol = region_returns[crisis_indicator == 0].std() * np.sqrt(252) * 100
        crisis_vol = region_returns[crisis_indicator == 1].std() * np.sqrt(252) * 100
        
        regional_impact[region.upper()] = {
            'Normal Return (%)': normal_ret,
            'Crisis Return (%)': crisis_ret,
            'Impact (%)': crisis_ret - normal_ret,
            'Normal Vol (%)': normal_vol,
            'Crisis Vol (%)': crisis_vol
        }

regional_df = pd.DataFrame(regional_impact).T
print("\nRegional Impact Analysis")
print(regional_df.round(2))

# Visualize regional impact
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Returns impact
regional_df['Impact (%)'].plot(kind='bar', ax=axes[0], color='coral', alpha=0.7)
axes[0].set_title('Regional Impact on Returns', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Return Impact (%)')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Volatility impact
vol_impact = regional_df[['Normal Vol (%)', 'Crisis Vol (%)']]
vol_impact.plot(kind='bar', ax=axes[1], color=['blue', 'darkred'], alpha=0.7)
axes[1].set_title('Regional Volatility: Normal vs Crisis', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Volatility (%)')
axes[1].legend(['Normal', 'Crisis'])
axes[1].grid(True, alpha=0.3)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 5. Crisis Event Case Studies

In [ ]:
# Analyze specific crisis events
crisis_events = config['backtest']['crisis_events']

for event in crisis_events:
    start = pd.to_datetime(event['start'])
    end = pd.to_datetime(event['end'])
    
    print(f"\n{'='*60}")
    print(f"EVENT: {event['name']}")
    print(f"Period: {start.date()} to {end.date()}")
    print(f"{'='*60}")
    
    # Filter data for event period
    event_mask = (equity_returns.index >= start) & (equity_returns.index <= end)
    
    if event_mask.sum() > 0:
        event_returns = equity_returns[event_mask].mean() * 252 * 100
        
        # Top winners and losers
        print("\nTop 5 Winners:")
        print(event_returns.nlargest(5).round(2))
        
        print("\nTop 5 Losers:")
        print(event_returns.nsmallest(5).round(2))
        
        # Shipping impact
        event_shipping = shipping_analyzed[event_mask]
        avg_reduction = event_shipping['reduction_pct'].mean() * 100
        print(f"\nAverage shipping reduction: {avg_reduction:.1f}%")
    else:
        print("No data available for this period")

## 6. Correlation Analysis

In [ ]:
# Calculate correlations with shipping disruption
corr_data = equity_returns.copy()
corr_data['shipping_disruption'] = shipping_analyzed['traffic_signal']
corr_data['geo_risk'] = geo_analyzed['composite_risk_score_smooth']

# Correlation with shipping disruption
shipping_corr = corr_data.corr()['shipping_disruption'].drop(['shipping_disruption', 'geo_risk']).sort_values()

print("\nCorrelation with Shipping Disruption:")
print("\nMost Negative (Hurt by Crisis):")
print(shipping_corr.head(5).round(3))
print("\nMost Positive (Benefit from Crisis):")
print(shipping_corr.tail(5).round(3))

# Visualize correlations
fig, ax = plt.subplots(figsize=(12, 8))
shipping_corr.plot(kind='barh', ax=ax, color=['red' if x < 0 else 'green' for x in shipping_corr])
ax.set_title('Asset Correlation with Shipping Disruption', fontsize=16, fontweight='bold')
ax.set_xlabel('Correlation Coefficient')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Key Insights Summary

In [ ]:
print("\n" + "="*70)
print("KEY INSIGHTS - MARKET RESPONSE TO STRAIT OF HORMUZ DISRUPTIONS")
print("="*70)

print("\n1. REGIONAL IMPACT RANKING (by return impact):")
regional_ranking = regional_df['Impact (%)'].sort_values()
for region, impact in regional_ranking.items():
    print(f"   {region}: {impact:+.2f}% {'(NEGATIVE)' if impact < 0 else '(POSITIVE)'}")

print("\n2. BEST PERFORMING ASSETS DURING CRISES:")
if len(asset_comp) > 0:
    best_assets = asset_comp['difference'].nlargest(3)
    for asset, diff in best_assets.items():
        print(f"   {asset}: +{diff:.2f}% outperformance")

print("\n3. WORST PERFORMING ASSETS DURING CRISES:")
if len(asset_comp) > 0:
    worst_assets = asset_comp['difference'].nsmallest(3)
    for asset, diff in worst_assets.items():
        print(f"   {asset}: {diff:.2f}% underperformance")

print("\n4. VOLATILITY CHANGES:")
for region, row in regional_df.iterrows():
    vol_change = ((row['Crisis Vol (%)'] / row['Normal Vol (%)']) - 1) * 100
    print(f"   {region}: +{vol_change:.1f}% increase in volatility")

print("\n5. TRADING STRATEGY IMPLICATIONS:")
print("   ✓ LONG: Energy equities, US Treasuries, Oil exporter currencies")
print("   ✓ SHORT: Asian equities, Transportation, Oil importer currencies")
print("   ✓ HEDGE: Increase volatility exposure, reduce leverage")

print("\n" + "="*70)